# Notebook 38 — Evaluation Fundamentals and Regression Testing

    ## Learning objectives

    - Turn product requirements into representative datasets and graders
- Combine deterministic, statistical, retrieval, model-based, and human evaluation
- Compare systems with slices, uncertainty, and regression gates

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 38.1 Begin with decisions

An evaluation is evidence for a decision: ship a prompt, choose a model, accept a
training run, or diagnose a failure. Define the unit, population, success criterion,
severity, and acceptable tradeoffs first. Freeze a representative test set; maintain a
separate development set; add production failures without repeatedly tuning on the test.


In [ ]:
examples = [
    {"id": "a", "slice": "arithmetic", "reference": "42", "output": "42"},
    {"id": "b", "slice": "arithmetic", "reference": "17", "output": "The answer is 17."},
    {"id": "c", "slice": "abstain", "reference": "INSUFFICIENT", "output": "Paris"},
]
def exact(reference, output): return float(reference.strip() == output.strip())
def contains(reference, output): return float(reference.lower() in output.lower())
for row in examples:
    row["exact"] = exact(row["reference"], row["output"])
    row["contains"] = contains(row["reference"], row["output"])
    print(row)


## 38.2 Match graders to failure modes

Use executable tests for code, schema validators for structure, exact/set comparison for
constrained answers, retrieval metrics for ranking, and expert review for domain nuance.
Semantic similarity is not factuality. LLM graders are scalable but biased and noisy.
Human labels also require guidelines and agreement checks. Prefer a portfolio of graders.


In [ ]:
import math, random
scores = [r["contains"] for r in examples]
mean = sum(scores) / len(scores)
# Tiny bootstrap demonstration; real sets need far more examples.
rng = random.Random(42)
boots = [sum(rng.choices(scores, k=len(scores))) / len(scores) for _ in range(5000)]
lo, hi = sorted(boots)[125], sorted(boots)[4874]
print(f"score={mean:.3f}, illustrative bootstrap interval=({lo:.3f}, {hi:.3f})")


## 38.3 Comparisons and gates

Pair outputs by example when comparing systems. Report delta and confidence interval,
not two isolated averages. Slice by language, length, risk, source, tool, and no-answer
status; averages hide regressions. A CI gate should require critical tests, protect key
slices, bound cost/latency, and store model revision, prompt, decoding, environment, raw
outputs, grader version, and dataset fingerprint.


## 38.4 From product contract to evaluation record

Write requirements as observable claims. “Helpful” is underspecified; “returns the correct
account policy, cites the effective policy version, and abstains when no policy applies” yields
separable graders. Define input distribution, expected output/behavior, allowed variation,
severity, latency/cost constraints, and action after failure. Evaluation examples should store
stable IDs, inputs/context, references or grading metadata, slices, provenance, and annotation
status—not only prompt/answer pairs.

Development examples guide iteration; a frozen test estimates generalization; a shadow set can
monitor production drift. Prevent contamination by grouping related sources and limiting repeated
inspection of final tests. Add discovered production failures to a regression set, but also sample
broadly so metrics do not become a catalog of yesterday's bugs. Version datasets and graders
independently; a score without both versions is not reproducible.


In [ ]:
# A compact evaluation-record schema and validation example.
from pydantic import BaseModel, ConfigDict, Field
class EvalRecord(BaseModel):
    model_config = ConfigDict(extra="forbid")
    id: str
    input: str
    reference: str | None = None
    slices: list[str] = Field(default_factory=list)
    severity: str = "normal"
    metadata: dict = Field(default_factory=dict)

record = EvalRecord(id="rag-001", input="What is the refund window?",
    reference="30 days", slices=["answerable", "policy"],
    metadata={"supporting_source": "refund-policy-v3"})
print(record.model_dump_json(indent=2))


## 38.5 Grader design and aggregation

Deterministic graders are preferred when the contract is deterministic: JSON/schema, regex,
set equality, numeric tolerance, executable unit tests, tool name/arguments, citation IDs, and
forbidden actions. Text similarity measures lexical/semantic closeness but not correctness.
Model graders handle nuance at higher variance/bias. Human review handles ambiguity and high
stakes but needs instructions, training, blinding, overlap, adjudication, and agreement analysis.

Avoid averaging incommensurate metrics into one opaque score. Use gates for critical invariants,
then report a metric vector and slices. If a weighted score is necessary, justify weights using
business cost and show components. Macro averaging weights groups/classes equally; micro averaging
weights examples/events. Decide how partial credit, multiple references, ties, abstentions, parser
failures, and grader errors are counted before viewing system results.


In [ ]:
# Slice report with Wilson intervals for binary outcomes.
from collections import defaultdict
def wilson(successes, n, z=1.96):
    if not n: return (float("nan"), float("nan"))
    p = successes/n; den = 1 + z*z/n
    center = (p + z*z/(2*n))/den
    radius = z*((p*(1-p)/n + z*z/(4*n*n))**.5)/den
    return center-radius, center+radius
outcomes = [("short", 1), ("short", 1), ("short", 0),
            ("long", 1), ("long", 0), ("long", 0)]
grouped = defaultdict(list)
for group, value in outcomes: grouped[group].append(value)
for group, values in grouped.items():
    lo, hi = wilson(sum(values), len(values))
    print(group, f"{sum(values)/len(values):.1%}", f"95% CI [{lo:.1%}, {hi:.1%}]")


## 38.6 Comparative statistics and experiment discipline

Evaluate systems on the same examples and use paired differences. Bootstrap examples (or
independent clusters such as users/documents) to estimate uncertainty. For binary paired
outcomes, inspect discordant pairs and consider McNemar-style analysis. Multiple experiments
and metric fishing inflate false discoveries; predeclare primary metrics and retain all results.
Statistical significance does not imply practical significance—set a minimum meaningful effect.

Stochastic generation adds within-example variation. Decide whether the estimand is expected
quality over sampling, pass@k, worst-case, or one production run; repeat seeds accordingly.
Model-based graders add grader variation and should be sampled/calibrated too. Keep raw outputs
to rerun new graders without repaying generation cost. Human adjudication should be blind to
system identity and randomized in presentation order.


In [ ]:
# Paired bootstrap for the accuracy difference between systems A and B.
import random, numpy as np
a = np.array([1,1,0,1,0,1,0,0,1,1])
b = np.array([1,0,0,1,1,1,0,0,0,1])
observed = float((a-b).mean())
rng = random.Random(7)
deltas = []
for _ in range(10_000):
    idx = [rng.randrange(len(a)) for _ in a]
    deltas.append(float((a[idx]-b[idx]).mean()))
lo, hi = np.quantile(deltas, [.025,.975])
print(f"paired delta={observed:+.3f}; bootstrap 95% CI [{lo:+.3f}, {hi:+.3f}]")


## 38.7 Evaluation operations reference

An experiment artifact should include application code commit, model ID/revision/provider,
prompt/template/tool schemas, decoding and seed, retrieval/index versions, dataset fingerprint,
grader versions/prompts/models, raw outputs/traces, environment, latency/tokens/cost, and summary
tables. Cache only when these inputs match. CI uses a small fast critical/regression suite;
scheduled runs cover broader/stochastic/costly cases; canary/shadow monitoring validates live
distributions without allowing unreviewed automated actions.

Common traps: test-set overfitting; leakage; only easy/answerable examples; unlabeled slice gaps;
references with errors; semantic similarity as factuality; ignoring parser/grader failures;
changing model and prompt simultaneously; averages without denominators/uncertainty; judging
only final answer when tools/retrieval can cause harm; and optimizing a proxy after it stops
tracking user value.

Use evaluation diagnostically: preserve failure clusters and traces, form a hypothesis, change
one component, rerun paired examples and broad regressions, then document the decision.


## 38.8 Evaluation reference

| Evaluation layer | Examples |
|---|---|
| Input/data | schema, distribution, leakage, slice coverage |
| Component | retrieval recall, tool args, parser validity |
| Output | correctness, constraints, citations, style |
| Trajectory/effects | calls, authorization, side effects, step budget |
| Operational | TTFT/latency, tokens, cost, error/fallback |
| Human/product | preference, task completion, escalation, satisfaction |

Always state numerator, denominator, unit of sampling, aggregation, and uncertainty. Pair comparisons
on the same examples. Separate critical release gates from optimization metrics. Store raw outputs and
grader details. Treat parse/timeouts/refusals as outcomes according to predeclared policy, not missing
data deleted after the fact.

Evaluation is iterative but protect a final test from overfitting. Add production failures to regression
coverage while refreshing broad representative samples. A metric becomes unreliable when optimized
without checking its relationship to user value—keep qualitative review and multiple independent
signals.


## 38.9 Confidence intervals and paired decisions

Evaluation examples are a sample from a target population. Report counts and uncertainty, preferably using paired bootstrap or randomization when comparing two systems on the same items. Do not treat repeated generations from one prompt as independent user cases. Predefine the minimum practically important difference and critical slice gates. Statistical significance cannot rescue an irrelevant benchmark, contaminated data, or a change that violates safety requirements.


In [ ]:
import random
diffs=[1,0,0,-1,1,1,0,-1]; rng=random.Random(4); means=[sum(rng.choices(diffs,k=len(diffs)))/len(diffs) for _ in range(2000)]; print("delta",sum(diffs)/len(diffs),"95% interval",sorted(means)[50],sorted(means)[1950])


## 38.10 Build a release matrix

A serious suite crosses capabilities and risks with data slices and system configurations. Include deterministic unit tests, task datasets, adversarial cases, calibration, latency/cost, and qualitative review. Store per-example predictions, errors, prompt/model/data revisions, and environment metadata. Promotion rules distinguish hard gates from monitored regressions and document overrides. Run the parent and candidate through the identical harness; never compare a fresh score with an undocumented historical number.


In [ ]:
matrix={"correctness":{"overall":.84,"min":.80},"schema":{"overall":.99,"min":.98},"critical_security":{"count":0,"max":0},"p95":{"value":2.4,"max":3.0}}; print(matrix); assert all([matrix["correctness"]["overall"]>=matrix["correctness"]["min"],matrix["critical_security"]["count"]<=matrix["critical_security"]["max"]])


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [Hugging Face Evaluate](https://huggingface.co/docs/evaluate/index)
- [Bootstrap methods](https://doi.org/10.1214/aos/1176344552)


## Exercises

    1. Write an evaluation contract for a RAG assistant before choosing metrics.
2. Implement recall@k, citation precision, and an abstention metric.
3. Compare two systems with a paired bootstrap and slice table.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
